# Exploratory Analysis

> **Note:** This notebook is exploratory and is not part of the reproducible pipeline.  
> It is committed for reference only — figures here are not final.  
> For final, reproducible figures, run `python scripts/run_all.py`.

---

In [ ]:
import sys
from pathlib import Path

# Add project root to path so src/ is importable
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_config, load_dataset, get_features_and_target
from src.data import describe_dataset, make_train_test_split
from src.metrics import compute_all_metrics, precision_at_recall
from src.models import train_logistic_regression, train_random_forest, get_scores

cfg = load_config()
print('Config loaded. Random seed:', cfg['random_seed'])

## 1. Dataset Exploration

In [ ]:
df = load_dataset(cfg, source='synthetic')
X, y = get_features_and_target(df)
stats = describe_dataset(y, label='synthetic')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
axes[0].bar(['Legitimate', 'Fraud'], [stats['n_legitimate'], stats['n_fraud']],
            color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')

# Feature distributions (first 4 features)
feature_cols = [c for c in df.columns if c != 'Class']
for i, col in enumerate(feature_cols[:4]):
    axes[1].hist(df[df['Class'] == 0][col], bins=40, alpha=0.5, label='Legit', density=True)
    axes[1].hist(df[df['Class'] == 1][col], bins=40, alpha=0.5, label='Fraud', density=True)
axes[1].set_title(f'Feature Distributions (first 4)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Quick Model Fit and Metric Overview

In [ ]:
X_train, X_test, y_train, y_test = make_train_test_split(X, y, cfg)

lr = train_logistic_regression(X_train, y_train, cfg)
rf = train_random_forest(X_train, y_train, cfg)

scores_lr = get_scores(lr, X_test)
scores_rf = get_scores(rf, X_test)

m_lr = compute_all_metrics(y_test, scores_lr)
m_rf = compute_all_metrics(y_test, scores_rf)

metrics_of_interest = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'auc_pr']
print(f"{'Metric':<12} {'Logistic Reg':>14} {'Random Forest':>14}")
print('-' * 42)
for k in metrics_of_interest:
    print(f"{k:<12} {m_lr[k]:>14.4f} {m_rf[k]:>14.4f}")

## 3. Precision@Recall — Quick Look

In [ ]:
recall_targets = cfg['recall_targets']
print(f"Precision@r for Random Forest:")
for r in recall_targets:
    prec, thr = precision_at_recall(y_test, scores_rf, recall_target=r)
    print(f"  P@{r:.0%} recall : Precision={prec:.4f}  Threshold={thr:.4f}")

---

*For the full, reproducible experiments with publication-quality figures, run:*

```bash
python scripts/run_all.py
```